# Classification de Tumeurs Cérébrales par IRM

Extraction de caractéristiques manuelles (histogramme, texture, contours Laplacian) + classification KNN.

**Dataset :** Brain Tumor MRI Dataset — 4 classes : `glioma`, `meningioma`, `notumor`, `pituitary`  
**Structure :** `dataset/Training/<classe>/` et `dataset/Testing/<classe>/`

In [ ]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, classification_report)
from tqdm import tqdm

## Section 1 — Imports & Configuration

In [ ]:
IMG_SIZE  = 128
TRAIN_DIR = "./dataset/Training"
TEST_DIR  = "./dataset/Testing"
CLASSES   = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_IDX = {name: i for i, name in enumerate(CLASSES)}

print("Configuration chargée.")
print(f"  Classes   : {CLASSES}")
print(f"  IMG_SIZE  : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Train dir : {TRAIN_DIR}")
print(f"  Test  dir : {TEST_DIR}")

## Section 2 — Exploration des données

In [ ]:
train_counts = [len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASSES]
test_counts  = [len(os.listdir(os.path.join(TEST_DIR,  c))) for c in CLASSES]

x = np.arange(len(CLASSES))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, train_counts, width, label="Train", color="steelblue")
ax.bar(x + width/2, test_counts,  width, label="Test",  color="coral")
ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylabel("Nombre d'images")
ax.set_title("Distribution des classes (Train vs Test)")
ax.legend(); plt.tight_layout(); plt.show()

print("Train:", dict(zip(CLASSES, train_counts)))
print("Test :", dict(zip(CLASSES, test_counts)))
print(f"Total : {sum(train_counts)+sum(test_counts)} images")

## Section 3 — Prétraitement

Pour chaque image :
1. Lecture depuis le sous-dossier de sa classe
2. Conversion en **niveaux de gris**
3. Redimensionnement à **128×128 pixels**
4. **Normalisation** des valeurs de pixels dans [0, 1] (division par 255)
5. Stockage dans des tableaux NumPy

In [ ]:
def load_images(root_dir):
    images, labels = [], []
    for cls in CLASSES:
        folder = os.path.join(root_dir, cls)
        label  = CLASS_IDX[cls]
        files  = [f for f in os.listdir(folder)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        for fname in tqdm(files, desc=f"  {cls}", leave=False):
            path = os.path.join(folder, fname)
            img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0          # normalize to [0, 1]
            images.append(img)
            labels.append(label)
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)


print("Chargement et prétraitement — Train...")
X_train_img, y_train = load_images(TRAIN_DIR)
print("Chargement et prétraitement — Test...")
X_test_img,  y_test  = load_images(TEST_DIR)

print(f"\nX_train_img : {X_train_img.shape}  (float32, valeurs ∈ [0,1])")
print(f"X_test_img  : {X_test_img.shape}")
print(f"y_train     : {y_train.shape}  — classes : {np.unique(y_train)}")
print(f"y_test      : {y_test.shape}")

## Section 4 — Extraction de caractéristiques

Chaque image est représentée par un **vecteur de 517 features** :

| Groupe | Description | Taille |
|---|---|---|
| **Histogramme** | Distribution des niveaux de gris (256 bins) | 256 |
| **Texture** | Variance, énergie, entropie, contraste, homogénéité | 5 |
| **Contours (Laplacian)** | Histogramme de la carte de gradients Laplacian | 256 |

> Le **Laplacian** détecte les bords en calculant la dérivée seconde de l'intensité — plus sensible aux détails fins que Canny.

In [ ]:
# ── Histogramme ──────────────────────────────────────────────────────────────
def histo(img):
    """Histogramme 256 bins sur l'image normalisée remappée en [0,255]."""
    img_u8 = (img * 255).astype(np.uint8)
    h, _   = np.histogram(img_u8.ravel(), bins=256, range=(0, 256))
    return h.astype(np.float64)


# ── Texture ───────────────────────────────────────────────────────────────────
def variance2(img):
    return np.var(img)

def energie(img):
    return np.sum(img ** 2) / img.size

def entropie(img):
    h, _ = np.histogram((img * 255).astype(np.uint8).ravel(),
                         bins=256, range=(0, 256))
    p = h / (h.sum() + 1e-10)
    return -np.sum(p * np.log2(p + 1e-10))

def contraste(img):
    return float(np.mean(np.abs(img[:, 1:] - img[:, :-1])))

def homogenite(img):
    return 1.0 / (1.0 + np.std(img))

def extract_texture(img):
    return np.array([variance2(img), energie(img), entropie(img),
                     contraste(img), homogenite(img)])


# ── Contours Laplacian ────────────────────────────────────────────────────────
def extract_edges(img):
    """Applique le filtre Laplacian et retourne l'histogramme de la carte de bords."""
    img_u8  = (img * 255).astype(np.uint8)
    lap     = cv2.Laplacian(img_u8, cv2.CV_64F)
    lap_u8  = np.clip(np.abs(lap), 0, 255).astype(np.uint8)
    h, _    = np.histogram(lap_u8.ravel(), bins=256, range=(0, 256))
    return h.astype(np.float64)


# ── Vecteur complet ───────────────────────────────────────────────────────────
def extract_features(img):
    """Concatène histogramme (256) + texture (5) + edges Laplacian (256) = 517."""
    return np.concatenate([histo(img), extract_texture(img), extract_edges(img)])


# ── Extraction sur tout le dataset ───────────────────────────────────────────
print("Extraction des features — Train...")
X_train = np.array([extract_features(img) for img in tqdm(X_train_img)])

print("Extraction des features — Test...")
X_test  = np.array([extract_features(img) for img in tqdm(X_test_img)])

print(f"\nTaille du vecteur de features : {X_train.shape[1]}")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

## Section 5 — Classification KNN

On utilise **K-Nearest Neighbors** avec K=5 et la distance euclidienne.  
Le modèle est entraîné sur les features extraites du jeu d'entraînement.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean", n_jobs=-1)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("Modèle KNN (K=5) entraîné et prédictions effectuées.")

## Section 6 — Évaluation du modèle

Métriques calculées sur le jeu de test (1 600 images) :
- **Accuracy** : taux global de bonnes classifications
- **Precision** : parmi les images prédites classe X, combien sont correctes
- **Recall** : parmi les vraies images classe X, combien sont détectées
- **F1-score** : moyenne harmonique de precision et recall

In [ ]:
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_test,   y_pred, average="macro", zero_division=0)
f1   = f1_score(y_test,       y_pred, average="macro", zero_division=0)

print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f} %)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-score  : {f1:.4f}")
print("=" * 45)
print()
print(classification_report(y_test, y_pred, target_names=CLASSES, zero_division=0))

## Section 7 — Matrice de confusion

Chaque ligne = classe réelle, chaque colonne = classe prédite.  
La diagonale représente les **bonnes classifications**.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.5, linecolor="gray")
plt.xlabel("Classe prédite", fontsize=12)
plt.ylabel("Classe réelle",  fontsize=12)
plt.title("Matrice de confusion — KNN (K=5)", fontsize=13)
plt.tight_layout()
plt.show()

## Section 8 — Visualisation des exemples

Un exemple d'IRM par classe issu du jeu d'entraînement,  
affiché après prétraitement (niveaux de gris, 128×128, normalisé).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, cls in zip(axes, CLASSES):
    # Prendre le premier index correspondant à cette classe
    idx = np.where(y_train == CLASS_IDX[cls])[0][0]
    img = X_train_img[idx]          # already grayscale, 128x128, [0,1]
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(cls, fontsize=13, fontweight="bold")
    ax.axis("off")

plt.suptitle("Exemple d'IRM par classe (après prétraitement)", fontsize=13)
plt.tight_layout()
plt.show()